## Section 1 — Setup & Loading

In [1]:
import pandas as pd
import ast
import os

# Paths
DATA_RAW       = '../data/raw/'
DATA_PROCESSED = '../data/processed/'
os.makedirs(DATA_PROCESSED, exist_ok=True)

# Load raw files
movies   = pd.read_csv(DATA_RAW + 'movies_metadata.csv', low_memory=False)
credits  = pd.read_csv(DATA_RAW + 'credits.csv')
keywords = pd.read_csv(DATA_RAW + 'keywords.csv')
links    = pd.read_csv(DATA_RAW + 'links.csv')
ratings  = pd.read_csv(DATA_RAW + 'ratings.csv')

print("movies_metadata :", movies.shape)
print("credits         :", credits.shape)
print("keywords        :", keywords.shape)
print("links           :", links.shape)
print("ratings         :", ratings.shape)

movies_metadata : (45466, 24)
credits         : (45476, 3)
keywords        : (46419, 2)
links           : (45843, 3)
ratings         : (26024289, 4)


## Section 2 — Fix Data Types

In [2]:
# Convert numeric columns
movies['budget']       = pd.to_numeric(movies['budget'], errors='coerce')
movies['revenue']      = pd.to_numeric(movies['revenue'], errors='coerce')
movies['popularity']   = pd.to_numeric(movies['popularity'], errors='coerce')
movies['runtime']      = pd.to_numeric(movies['runtime'], errors='coerce')
movies['vote_average'] = pd.to_numeric(movies['vote_average'], errors='coerce')
movies['vote_count']   = pd.to_numeric(movies['vote_count'], errors='coerce')

# Convert id to integer for joining
movies['id'] = pd.to_numeric(movies['id'], errors='coerce')
movies = movies.dropna(subset=['id'])
movies['id'] = movies['id'].astype(int)

# Parse release date and extract year and decade
movies['release_date'] = pd.to_datetime(movies['release_date'], errors='coerce')
movies['year']         = movies['release_date'].dt.year
movies['decade']       = (movies['year'] // 10) * 10

print("Data types after conversion:")
print(movies[['budget', 'revenue', 'popularity', 'runtime', 'vote_average', 'vote_count', 'id', 'year', 'decade']].dtypes)
print("\nShape after id cleaning:", movies.shape)

Data types after conversion:
budget          float64
revenue         float64
popularity      float64
runtime         float64
vote_average    float64
vote_count      float64
id                int64
year            float64
decade          float64
dtype: object

Shape after id cleaning: (45463, 26)


In [4]:
# Replace zero budget and revenue with NaN
movies.loc[movies['budget'] == 0, 'budget']   = None
movies.loc[movies['revenue'] == 0, 'revenue'] = None

print("Remaining valid values after cleaning:")
print("Budget    :", movies['budget'].notna().sum())
print("Revenue   :", movies['revenue'].notna().sum())

Remaining valid values after cleaning:
Budget    : 8890
Revenue   : 7408


## Section 4 — Parse JSON Columns


In [7]:
# Helper function to parse stringified JSON columns
def parse_json_column(val, key):
    try:
        items = ast.literal_eval(val)
        return [item[key] for item in items]
    except:
        return []

# Parse genres
movies['genres_list'] = movies['genres'].apply(lambda x: parse_json_column(x, 'name'))

# Parse production countries
movies['countries_list'] = movies['production_countries'].apply(lambda x: parse_json_column(x, 'name'))

# Parse production companies
movies['companies_list'] = movies['production_companies'].apply(lambda x: parse_json_column(x, 'name'))

# Parse spoken languages
movies['languages_list'] = movies['spoken_languages'].apply(lambda x: parse_json_column(x, 'name'))

print("Sample genres_list:")
print(movies[['title', 'genres_list', 'countries_list', 'companies_list', 'languages_list']].head(3))

Sample genres_list:
              title                   genres_list              countries_list  \
0         Toy Story   [Animation, Comedy, Family]  [United States of America]   
1           Jumanji  [Adventure, Fantasy, Family]  [United States of America]   
2  Grumpier Old Men             [Romance, Comedy]  [United States of America]   

                                      companies_list       languages_list  
0                          [Pixar Animation Studios]            [English]  
1  [TriStar Pictures, Teitler Film, Interscope Co...  [English, Français]  
2                     [Warner Bros., Lancaster Gate]            [English]  


## Section 5 — Extract Directors and top cast from Credits


In [8]:
# Parse crew column and extract director
def extract_director(crew_str):
    try:
        crew = ast.literal_eval(crew_str)
        for member in crew:
            if member['job'] == 'Director':
                return member['name']
    except:
        return None
    return None

credits['director'] = credits['crew'].apply(extract_director)

# Convert id to integer for joining
credits['id'] = pd.to_numeric(credits['id'], errors='coerce')
credits = credits.dropna(subset=['id'])
credits['id'] = credits['id'].astype(int)

# Merge director into movies
movies = movies.merge(credits[['id', 'director']], on='id', how='left')

print("Director column sample:")
print(movies[['title', 'director']].head(10))
print("\nMissing directors:", movies['director'].isna().sum())

Director column sample:
                         title         director
0                    Toy Story    John Lasseter
1                      Jumanji     Joe Johnston
2             Grumpier Old Men    Howard Deutch
3            Waiting to Exhale  Forest Whitaker
4  Father of the Bride Part II    Charles Shyer
5                         Heat     Michael Mann
6                      Sabrina   Sydney Pollack
7                 Tom and Huck     Peter Hewitt
8                 Sudden Death      Peter Hyams
9                    GoldenEye  Martin Campbell

Missing directors: 888


In [11]:
# Extract top 3 billed actors from cast JSON
def extract_cast(cast_str, n=3):
    try:
        cast = ast.literal_eval(cast_str)
        cast_sorted = sorted(cast, key=lambda x: x['order'])
        return [member['name'] for member in cast_sorted[:n]]
    except:
        return []

credits['top_cast'] = credits['cast'].apply(extract_cast)

# Merge top cast into movies
movies = movies.merge(credits[['id', 'top_cast']], on='id', how='left')

print("Sample cast:")
print(movies[['title', 'director', 'top_cast']].head(10))
print("\nMissing cast:", movies['top_cast'].isna().sum())

Sample cast:
                         title         director  \
0                    Toy Story    John Lasseter   
1                      Jumanji     Joe Johnston   
2             Grumpier Old Men    Howard Deutch   
3            Waiting to Exhale  Forest Whitaker   
4  Father of the Bride Part II    Charles Shyer   
5                         Heat     Michael Mann   
6                      Sabrina   Sydney Pollack   
7                 Tom and Huck     Peter Hewitt   
8                 Sudden Death      Peter Hyams   
9                    GoldenEye  Martin Campbell   

                                            top_cast  
0                [Tom Hanks, Tim Allen, Don Rickles]  
1     [Robin Williams, Jonathan Hyde, Kirsten Dunst]  
2         [Walter Matthau, Jack Lemmon, Ann-Margret]  
3  [Whitney Houston, Angela Bassett, Loretta Devine]  
4         [Steve Martin, Diane Keaton, Martin Short]  
5            [Al Pacino, Robert De Niro, Val Kilmer]  
6        [Harrison Ford, Julia Ormond, G

## Section 6 — Build Working Dataframes

In [ ]:
# Drop raw and unused columns
movies_clean = movies.drop(columns=[
    'genres',
    'production_countries',
    'production_companies',
    'spoken_languages',
    'belongs_to_collection',
    'homepage',
    'poster_path',
    'adult',
    'video',
    'imdb_id',
    'tagline',
    'overview',
    'original_title',
    'status'
]).copy()

# movies_financial subset with valid budget and revenue
movies_financial = movies_clean[
    movies_clean['budget'].notna() &
    movies_clean['revenue'].notna()
].copy()

# Add ROI column
movies_financial['roi'] = movies_financial['revenue'] / movies_financial['budget']

print("movies_clean shape    :", movies_clean.shape)
print("movies_financial shape:", movies_financial.shape)
print("\nColumns in movies_clean:")
print(movies_clean.columns.tolist())

movies_clean shape    : (45697, 18)
movies_financial shape: (5417, 19)

Columns in movies_clean:
['budget', 'id', 'original_language', 'popularity', 'release_date', 'revenue', 'runtime', 'title', 'vote_average', 'vote_count', 'year', 'decade', 'genres_list', 'countries_list', 'companies_list', 'languages_list', 'director', 'top_cast']


## Section 7 — Build rating


In [ ]:
# Convert ids to integer for joining
links['tmdbId']  = pd.to_numeric(links['tmdbId'], errors='coerce')
links['movieId'] = pd.to_numeric(links['movieId'], errors='coerce')
links = links.dropna(subset=['tmdbId', 'movieId'])
links['tmdbId']  = links['tmdbId'].astype(int)

ratings['movieId'] = ratings['movieId'].astype(int)

# Join ratings with links to get tmdbId
ratings_linked = ratings.merge(links[['movieId', 'tmdbId']], on='movieId', how='inner')

# Join with movies_clean to get movie metadata
ratings_enriched = ratings_linked.merge(movies_clean, left_on='tmdbId', right_on='id', how='inner')
ratings_enriched = ratings_enriched.drop(columns=['movieId'])

print("ratings_enriched shape:", ratings_enriched.shape)
print("\nColumns:")
print(ratings_enriched.columns.tolist())
print("\nSample:")
print(ratings_enriched[['userId', 'rating', 'title', 'year', 'director', 'genres_list']].head(5))


## Section 8 — Enrich Movies with MovieLens Ratings


In [14]:
# Compute average rating and vote count per movie
movielens_stats = ratings_enriched.groupby('id').agg(
    movielens_avg_rating = ('rating', 'mean'),
    movielens_vote_count = ('rating', 'count')
).reset_index()

# Compute full ratings list per movie
movielens_ratings_list = ratings_enriched.groupby('id')['rating'].apply(list).reset_index()
movielens_ratings_list.columns = ['id', 'ratings_list']

# Merge both into movies_clean
movies_clean = movies_clean.merge(movielens_stats, on='id', how='left')
movies_clean = movies_clean.merge(movielens_ratings_list, on='id', how='left')

print("movies_clean shape:", movies_clean.shape)
print("\nColumns:", movies_clean.columns.tolist())
print("\nSample:")
print(movies_clean[['title', 'vote_average', 'movielens_avg_rating', 'movielens_vote_count']].dropna().head(5))

movies_clean shape: (45697, 21)

Columns: ['budget', 'id', 'original_language', 'popularity', 'release_date', 'revenue', 'runtime', 'title', 'vote_average', 'vote_count', 'year', 'decade', 'genres_list', 'countries_list', 'companies_list', 'languages_list', 'director', 'top_cast', 'movielens_avg_rating', 'movielens_vote_count', 'ratings_list']

Sample:
                         title  vote_average  movielens_avg_rating  \
0                    Toy Story           7.7              3.888157   
1                      Jumanji           6.9              3.236953   
2             Grumpier Old Men           6.5              3.175550   
3            Waiting to Exhale           6.1              2.875713   
4  Father of the Bride Part II           5.7              3.079565   

   movielens_vote_count  
0               66008.0  
1               26060.0  
2               15497.0  
3                2981.0  
4               15258.0  


## Section 9 — Save Processed Files


In [17]:
# Save final processed files
movies_clean.to_csv(DATA_PROCESSED + 'movies.csv', index=False)
ratings_enriched.to_csv(DATA_PROCESSED + 'ratings_enriched.csv', index=False)

print("movies.csv saved         :", DATA_PROCESSED + 'movies.csv')
print("ratings_enriched saved   :", DATA_PROCESSED + 'ratings_enriched.csv')
print("\nmovies.csv shape         :", movies_clean.shape)
print("ratings_enriched shape   :", ratings_enriched.shape)

movies.csv saved         : ../data/processed/movies.csv
ratings_enriched saved   : ../data/processed/ratings_enriched.csv

movies.csv shape         : (45697, 21)
ratings_enriched shape   : (26049433, 23)


## Section 10 — Summary


### movies.csv — 45,697 rows, 21 columns

One row per movie. Final clean dataset used for all visualizations.

| Column | Description |
|---|---|
| id | TMDB movie ID |
| title | Movie title |
| year | Release year |
| decade | Release decade |
| release_date | Full release date |
| budget | Production budget in USD (NaN if unknown) |
| revenue | Box office revenue in USD (NaN if unknown) |
| runtime | Runtime in minutes |
| original_language | Original language code (en, fr, etc.) |
| popularity | TMDB popularity score |
| vote_average | TMDB average rating (0-10) |
| vote_count | Number of TMDB votes |
| genres_list | List of genres e.g. [Action, Comedy] |
| countries_list | List of production countries |
| companies_list | List of production companies |
| languages_list | List of spoken languages |
| director | Director name |
| top_cast | Top 3 billed actors as a list |
| movielens_avg_rating | Average MovieLens rating (0.5-5.0) |
| movielens_vote_count | Number of MovieLens ratings |
| ratings_list | Full list of individual MovieLens ratings |

### ratings_enriched.csv — 26,049,433 rows, 23 columns

One row per user-movie rating with full movie metadata attached.

| Column | Description |
|---|---|
| userId | Anonymous user ID |
| movieId | MovieLens movie ID |
| rating | User rating (0.5 to 5.0) |
| timestamp | Unix timestamp of the rating |
| tmdbId | TMDB movie ID |
| title, year, director, genres_list... | All columns from movies.csv |
